# **1. DATA SOURCE**

Mengambil data weather dari []

Mengambil data berita cuaca dari []

In [1]:
# ini buat requirement installation
 
# !pip install kafka-python requests feedparser
import json
import time
import hashlib
import threading
import requests
import feedparser
from datetime import datetime
from kafka import KafkaProducer, KafkaAdminClient, KafkaConsumer
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

In [2]:
#buat ambil data dari datasource (berita)

BOOTSTRAP_SERVERS = ["localhost:9092"]

KOTA = [
    {"kode": "JKT", "nama": "Jakarta",  "lat": -6.21, "lon": 106.85},
    {"kode": "SBY", "nama": "Surabaya", "lat": -7.25, "lon": 112.75},
    {"kode": "SMG", "nama": "Semarang", "lat": -6.99, "lon": 110.42},
    {"kode": "MDN", "nama": "Medan",    "lat": -3.59, "lon": 98.67},
    {"kode": "MKS", "nama": "Makassar", "lat": -5.14, "lon": 119.41},
    {"kode": "DPS", "nama": "Denpasar", "lat": -8.67, "lon": 115.21},
]

RSS_URLS = [
   "https://www.antaranews.com/rss/warta-bumi.xml",
"https://www.mongabay.co.id/feed/",
]

TOPIC_API = "weather-api"
TOPIC_RSS = "weather-rss"

INTERVAL_API = 600   # 10 menit
INTERVAL_RSS = 300   # 5 menit

# Stop flag — panggil stop_flag.set() dari sel manapun untuk hentikan semua thread
stop_flag = threading.Event()

print(" Konfigurasi selesai")
print(f"   Kota dipantau : {[k['kode'] for k in KOTA]}")
print(f"   Topic API     : {TOPIC_API}")
print(f"   Topic RSS     : {TOPIC_RSS}")

 Konfigurasi selesai
   Kota dipantau : ['JKT', 'SBY', 'SMG', 'MDN', 'MKS', 'DPS']
   Topic API     : weather-api
   Topic RSS     : weather-rss


In [3]:
#buat ambil data dari datasource (api cuaca) + tes koneksi API

def ambil_cuaca(kota: dict) -> dict | None:
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": kota["lat"],
        "longitude": kota["lon"],
        "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
        "timezone": "Asia/Jakarta",
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()["current"]
        return {
            "kode_kota": kota["kode"],
            "nama_kota": kota["nama"],
            "temperature": data["temperature_2m"],
            "humidity":    data["relative_humidity_2m"],
            "wind_speed":  data["wind_speed_10m"],
            "weather_code": data["weather_code"],
            "timestamp":   datetime.now().isoformat(),
        }
    except Exception as e:
        print(f"  [ERROR] {kota['nama']}: {e}")
        return None

# Test: ambil data semua kota sekali
print("Test fetch Open-Meteo:\n")
for kota in KOTA:
    event = ambil_cuaca(kota)
    if event:
        print(f"  {event['kode_kota']} | {event['nama_kota']:10s} | "
              f"{event['temperature']}°C | "
              f" {event['wind_speed']} km/h | "
              f" {event['humidity']}%")

Test fetch Open-Meteo:

  JKT | Jakarta    | 30.2°C |  7.6 km/h |  74%
  SBY | Surabaya   | 31.5°C |  12.6 km/h |  58%
  SMG | Semarang   | 31.8°C |  12.3 km/h |  57%
  MDN | Medan      | 27.1°C |  5.8 km/h |  81%
  MKS | Makassar   | 29.0°C |  4.3 km/h |  78%
  DPS | Denpasar   | 28.5°C |  15.8 km/h |  75%


In [4]:
# buat test koneksi RSS single fetch, tidak looping

feedparser.USER_AGENT = "Mozilla/5.0 (compatible; WeatherPulse/1.0)"

def hash_url(url: str) -> str:
    return hashlib.md5(url.encode()).hexdigest()[:8]

def fetch_rss_sekali(url: str) -> list[dict]:
    hasil = []
    try:
        feed = feedparser.parse(url)
        print(f"  Status  : {feed.get('status', 'N/A')}")
        print(f"  Feed    : {feed.feed.get('title', 'tidak ada judul')}")
        print(f"  Entries : {len(feed.entries)} artikel")
        for entry in feed.entries[:3]:  # tampilkan 3 pertama
            link = entry.get("link", "")
            artikel = {
                "judul":       entry.get("title", ""),
                "link":        link,
                "ringkasan":   entry.get("summary", "")[:200],
                "waktu_terbit": entry.get("published", datetime.now().isoformat()),
                "sumber":      feed.feed.get("title", url),
                "timestamp":   datetime.now().isoformat(),
            }
            hasil.append(artikel)
    except Exception as e:
        print(f"  [ERROR] {e}")
    return hasil

# Test semua RSS URL
for url in RSS_URLS:
    print(f"\nTest RSS: {url}")
    artikel = fetch_rss_sekali(url)
    for a in artikel:
        print(f"  → {a['judul'][:70]}...")


Test RSS: https://www.antaranews.com/rss/warta-bumi.xml
  Status  : 200
  Feed    : Warta Bumi - ANTARA News
  Entries : 20 artikel
  → Mengenal fenomena "Whirlpool" dan penyebab terjadinya...
  → Ingat, ini dampak buruk menyalakan kembang api bagi lingkungan...
  → Hati-hati, ini jenis ular yang pandai memanjat pohon...

Test RSS: https://www.mongabay.co.id/feed/
  Status  : 301
  Feed    : Mongabay.co.id
  Entries : 20 artikel
  → Pakis Ekor Monyet, Tumbuhan Unik di Gunung Muria...
  → Nasib Nelayan Ketika ‘Red Devil’ Kuasai Danau Batur...
  → Ular Terpanjang dalam Sejarah Bumi Ternyata Dua Kali Lipat Panjang Ibu...


# **2. DATA INGEST**

In [5]:
#buat masukin data yang diambil ke kafka dan tes koneksi Kafka

producer_api = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    enable_idempotence=True,
    acks="all",
)

def loop_producer_api():
    print("[API Producer] Dimulai")
    while not stop_flag.is_set():
        print(f"\n[API Producer] [{datetime.now().strftime('%H:%M:%S')}] Polling...")
        for kota in KOTA:
            if stop_flag.is_set():
                break
            event = ambil_cuaca(kota)
            if event:
                producer_api.send(TOPIC_API, key=kota["kode"], value=event)
                print(f"   {event['kode_kota']} | {event['temperature']}°C | "
                      f" {event['wind_speed']} km/h |  {event['humidity']}%")
        producer_api.flush()
        # tunggu interval, tapi cek stop_flag tiap 5 detik
        for _ in range(INTERVAL_API // 5):
            if stop_flag.is_set():
                break
            time.sleep(5)
    print("[API Producer] Berhenti")

thread_api = threading.Thread(target=loop_producer_api, daemon=True, name="ProducerAPI")
thread_api.start()
print(f" Thread producer API berjalan: {thread_api.is_alive()}")

[API Producer] Dimulai

[API Producer] [15:15:13] Polling...
 Thread producer API berjalan: True


In [6]:
# Buat topik Kafka jika belum ada sebelum Producer dijalankan
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP_SERVERS = ["localhost:9092"]

def buat_topik_jika_belum_ada(nama_topik: str):
    admin_client = KafkaAdminClient(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        client_id='test_admin_client'
    )
    
    topik_baru = NewTopic(name=nama_topik, num_partitions=1, replication_factor=1)
    
    try:
        admin_client.create_topics(new_topics=[topik_baru], validate_only=False)
        print(f"✅ Topik '{nama_topik}' berhasil dibuat.")
    except TopicAlreadyExistsError:
        print(f"ℹ️ Topik '{nama_topik}' sudah ada.")
    except Exception as e:
        print(f"❌ Gagal membuat topik '{nama_topik}': {e}")
    finally:
        admin_client.close()

# Buat topik yang dibutuhkan
buat_topik_jika_belum_ada(TOPIC_API)
buat_topik_jika_belum_ada(TOPIC_RSS)

ℹ️ Topik 'weather-api' sudah ada.
ℹ️ Topik 'weather-rss' sudah ada.


In [7]:
#buat masukin data yang diambil ke kafka dan tes koneksi Kafka

producer_rss = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    enable_idempotence=True,
    acks="all",
)

sudah_dikirim = set()  # deduplication

def loop_producer_rss():
    print("[RSS Producer] Dimulai")
    while not stop_flag.is_set():
        print(f"\n[RSS Producer] [{datetime.now().strftime('%H:%M:%S')}] Polling...")
        total_baru = 0
        for url in RSS_URLS:
            try:
                feed = feedparser.parse(url)
                for entry in feed.entries:
                    link = entry.get("link", "")
                    if not link or link in sudah_dikirim:
                        continue
                    artikel = {
                        "judul":        entry.get("title", ""),
                        "link":         link,
                        "ringkasan":    entry.get("summary", "")[:300],
                        "waktu_terbit": entry.get("published", datetime.now().isoformat()),
                        "sumber":       feed.feed.get("title", url),
                        "timestamp":    datetime.now().isoformat(),
                    }
                    key = hash_url(link)
                    producer_rss.send(TOPIC_RSS, key=key, value=artikel)
                    sudah_dikirim.add(link)
                    total_baru += 1
            except Exception as e:
                print(f"  [ERROR] RSS {url}: {e}")
        producer_rss.flush()
        print(f"  {total_baru} artikel baru dikirim ke {TOPIC_RSS}")
        for _ in range(INTERVAL_RSS // 5):
            if stop_flag.is_set():
                break
            time.sleep(5)
    print("[RSS Producer] Berhenti")

thread_rss = threading.Thread(target=loop_producer_rss, daemon=True, name="ProducerRSS")
thread_rss.start()
print(f" Thread producer RSS berjalan: {thread_rss.is_alive()}")

[RSS Producer] Dimulai

[RSS Producer] [15:15:13] Polling...
 Thread producer RSS berjalan: True


In [8]:
# buat verifikasi

from kafka import TopicPartition

def peek_topic(topic: str, n: int = 5):
    consumer = KafkaConsumer(
        bootstrap_servers=BOOTSTRAP_SERVERS,
        auto_offset_reset="earliest",
        consumer_timeout_ms=5000,
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        key_deserializer=lambda k: k.decode("utf-8") if k else None,
    )
    consumer.subscribe([topic])
    print(f"\n Isi topic '{topic}' (maks {n} event):\n")
    count = 0
    for msg in consumer:
        print(f"  offset={msg.offset} | key={msg.key} | "
              f"value={json.dumps(msg.value, ensure_ascii=False)[:120]}...")
        count += 1
        if count >= n:
            break
    if count == 0:
        print("  (kosong — tunggu producer polling pertama selesai)")
    consumer.close()

peek_topic(TOPIC_API)
peek_topic(TOPIC_RSS)


 Isi topic 'weather-api' (maks 5 event):

  offset=0 | key=JKT | value={"kode_kota": "JKT", "nama_kota": "Jakarta", "temperature": 30.6, "humidity": 69, "wind_speed": 7.8, "weather_code": 51,...
  offset=1 | key=SBY | value={"kode_kota": "SBY", "nama_kota": "Surabaya", "temperature": 31.9, "humidity": 54, "wind_speed": 11.0, "weather_code": 3...
  offset=2 | key=SMG | value={"kode_kota": "SMG", "nama_kota": "Semarang", "temperature": 32.0, "humidity": 57, "wind_speed": 11.5, "weather_code": 2...
  offset=3 | key=MDN | value={"kode_kota": "MDN", "nama_kota": "Medan", "temperature": 26.7, "humidity": 83, "wind_speed": 4.4, "weather_code": 53, "...
  offset=4 | key=MKS | value={"kode_kota": "MKS", "nama_kota": "Makassar", "temperature": 29.1, "humidity": 76, "wind_speed": 4.2, "weather_code": 3,...

 Isi topic 'weather-rss' (maks 5 event):

  offset=0 | key=d4b03acb | value={"judul": "Mengenal fenomena \"Whirlpool\" dan penyebab terjadinya", "link": "https://www.antaranews.com/berita/536

In [9]:
# buat cek status hidup/almarhum

threads = {
    "ProducerAPI": thread_api,
    "ProducerRSS": thread_rss,
}

print("Status thread:\n")
for nama, t in threads.items():
    status = "🟢 berjalan" if t.is_alive() else "🔴 mati"
    print(f"  {nama:15s} : {status}")

Status thread:

  ProducerAPI     : 🟢 berjalan
  ProducerRSS     : 🟢 berjalan


# **3. DATA STORE**

In [10]:
import os
import subprocess

FLUSH_INTERVAL = 120   # flush ke HDFS tiap 2 menit
HDFS_API_PATH  = "/data/weather/api"
HDFS_RSS_PATH  = "/data/weather/rss"

buffer_api = []
buffer_rss = []
lock_api   = threading.Lock()
lock_rss   = threading.Lock()

def simpan_ke_hdfs(data: list, hdfs_path: str, label: str):
    if not data:
        return
    ts        = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    tmp_local = f"/tmp/weather_{label}_{ts}.json"
    tmp_container = tmp_local
    hdfs_dest = f"{hdfs_path}/{ts}.json"

    # Tulis sementara ke lokal
    with open(tmp_local, "w") as f:
        json.dump(data, f, ensure_ascii=False)

    # Copy ke container → upload ke HDFS
    subprocess.run(["docker", "cp", tmp_local, f"hadoop-namenode:{tmp_container}"], capture_output=True)
    result = subprocess.run(
        ["docker", "exec", "hadoop-namenode",
         "hdfs", "dfs", "-put", "-f", tmp_container, hdfs_dest],
        capture_output=True, text=True
    )

    if result.returncode == 0:
        print(f"  [{label}] ✅ {hdfs_dest} ({len(data)} event)")
    else:
        print(f"  [{label}] ❌ Gagal: {result.stderr.strip()}")

    os.remove(tmp_local)

def pastikan_direktori_hdfs_ada(path: str):
    # Cek apakah direktori sudah ada
    cek = subprocess.run(["docker", "exec", "hadoop-namenode", "hdfs", "dfs", "-test", "-d", path])
    
    # Return code 0 berarti direktori exist
    if cek.returncode != 0:
        print(f"📂 {path} tidak ditemukan di HDFS. Membuat direktori baru...")
        buat = subprocess.run(["docker", "exec", "hadoop-namenode", "hdfs", "dfs", "-mkdir", "-p", path])
        if buat.returncode == 0:
            print(f"✅ {path} Berhasil dibuat.")
        else:
            print(f"❌ Gagal membuat {path}.")
    else:
        print(f"✅ {path} Sudah tersedia di HDFS, lanjut ke proses berikutnya.")

# Pastikan folder ada sebelum lanjut menjalankan hal lain
pastikan_direktori_hdfs_ada(HDFS_API_PATH)
pastikan_direktori_hdfs_ada(HDFS_RSS_PATH)


def loop_consumer_api():
    consumer = KafkaConsumer(
        TOPIC_API,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id="consumer-hdfs-api",
        auto_offset_reset="earliest",
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        consumer_timeout_ms=2000,
    )
    print("[Consumer API] Dimulai")
    last_flush = time.time()
    while not stop_flag.is_set():
        for msg in consumer:
            with lock_api:
                buffer_api.append(msg.value)
            if stop_flag.is_set():
                break
        if time.time() - last_flush >= FLUSH_INTERVAL:
            with lock_api:
                data_copy = buffer_api.copy()
                buffer_api.clear()
            simpan_ke_hdfs(data_copy, HDFS_API_PATH, "API")
            last_flush = time.time()
    # flush sisa saat stop
    with lock_api:
        data_copy = buffer_api.copy()
        buffer_api.clear()
    simpan_ke_hdfs(data_copy, HDFS_API_PATH, "API")
    consumer.close()
    print("[Consumer API] Berhenti")

def loop_consumer_rss():
    consumer = KafkaConsumer(
        TOPIC_RSS,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id="consumer-hdfs-rss",
        auto_offset_reset="earliest",
        value_deserializer=lambda m: json.loads(m.decode("utf-8")),
        consumer_timeout_ms=2000,
    )
    print("[Consumer RSS] Dimulai")
    last_flush = time.time()
    while not stop_flag.is_set():
        for msg in consumer:
            with lock_rss:
                buffer_rss.append(msg.value)
            if stop_flag.is_set():
                break
        if time.time() - last_flush >= FLUSH_INTERVAL:
            with lock_rss:
                data_copy = buffer_rss.copy()
                buffer_rss.clear()
            simpan_ke_hdfs(data_copy, HDFS_RSS_PATH, "RSS")
            last_flush = time.time()
    with lock_rss:
        data_copy = buffer_rss.copy()
        buffer_rss.clear()
    simpan_ke_hdfs(data_copy, HDFS_RSS_PATH, "RSS")
    consumer.close()
    print("[Consumer RSS] Berhenti")

thread_consumer_api = threading.Thread(target=loop_consumer_api, daemon=True, name="ConsumerAPI")
thread_consumer_rss = threading.Thread(target=loop_consumer_rss, daemon=True, name="ConsumerRSS")
thread_consumer_api.start()
thread_consumer_rss.start()

print(f" Consumer API berjalan : {thread_consumer_api.is_alive()}")
print(f" Consumer RSS berjalan : {thread_consumer_rss.is_alive()}")
print(f" Flush ke HDFS setiap {FLUSH_INTERVAL} detik")

   JKT | 30.2°C |  7.6 km/h |  74%
✅ /data/weather/api Sudah tersedia di HDFS, lanjut ke proses berikutnya.
   SBY | 31.5°C |  12.6 km/h |  58%
  40 artikel baru dikirim ke weather-rss
   SMG | 31.8°C |  12.3 km/h |  57%
✅ /data/weather/rss Sudah tersedia di HDFS, lanjut ke proses berikutnya.
 Consumer API berjalan : True
 Consumer RSS berjalan : True
 Flush ke HDFS setiap 120 detik
[Consumer API] Dimulai
[Consumer RSS] Dimulai


In [11]:
# Jalankan setelah 2 menit untuk confirm file masuk HDFS

def cek_hdfs(path):
    result = subprocess.run(
        ["docker", "exec", "hadoop-namenode", "hdfs", "dfs", "-ls", path],
        capture_output=True, text=True
    )
    print(f"📂 {path}:")
    print(result.stdout if result.stdout else "  (kosong — tunggu flush berikutnya)")

cek_hdfs("/data/weather/api")
cek_hdfs("/data/weather/rss")

   MDN | 27.1°C |  5.8 km/h |  81%
📂 /data/weather/api:
Found 6 items
-rw-r--r--   1 hadoop supergroup        991 2026-05-06 07:37 /data/weather/api/2026-05-06_14-37-55.json
-rw-r--r--   1 hadoop supergroup       1982 2026-05-06 07:45 /data/weather/api/2026-05-06_14-45-58.json
-rw-r--r--   1 hadoop supergroup       1651 2026-05-06 07:56 /data/weather/api/2026-05-06_14-56-00.json
-rw-r--r--   1 hadoop supergroup        331 2026-05-06 07:56 /data/weather/api/2026-05-06_14-56-25.json
-rw-r--r--   1 hadoop supergroup        990 2026-05-06 08:06 /data/weather/api/2026-05-06_15-06-04.json
-rw-r--r--   1 hadoop supergroup        990 2026-05-06 08:06 /data/weather/api/2026-05-06_15-06-29.json

   MKS | 29.0°C |  4.3 km/h |  78%
   DPS | 28.5°C |  15.8 km/h |  75%
📂 /data/weather/rss:
Found 2 items
-rw-r--r--   1 hadoop supergroup      23850 2026-05-06 07:37 /data/weather/rss/2026-05-06_14-37-55.json
-rw-r--r--   1 hadoop supergroup      23850 2026-05-06 07:46 /data/weather/rss/2026-05-06_14-45

# **4. DATA ANALYSIS**

### Objektif


  - 1. Perbandingan suhu antar kota: rata-rata, suhu tertinggi, terendah per kota — groupBy kode_kota
  - 2. Deteksi kondisi ekstrem: filter event dengan wind_speed > 40 km/h OR humidity > 90% OR temperature > 35°C — berapa     kali dan kota mana?
  - 3. Tren suhu per jam dalam sehari: rata-rata suhu semua kota per jam — apakah ada pola diurnal?

Fokus dashboard: Tabel suhu semua kota saat ini (warna berdasarkan suhu) · Highlight kota ekstrem · Berita cuaca terbaru

In [12]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import hour, avg, max, min, col, count, to_timestamp

# spark = SparkSession.builder \
#     .appName("WeatherPulseAnalysis") \
#     .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
#     .getOrCreate()

# print("Reading data from HDFS...")
# df_api = spark.read.option("multiLine", True).json("/data/weather/api/*.json")
os.environ["HADOOP_USER_NAME"] = "hadoop"
HDFS_URI = "hdfs://localhost:8020"

spark = SparkSession.builder \
    .appName("WeatherPulseAnalysis") \
    .config("spark.hadoop.fs.defaultFS", HDFS_URI) \
    .getOrCreate()

df_api = spark.read.option("multiLine", True).json(f"{HDFS_URI}/data/weather/api/")


df_api = df_api.withColumn("ts_obj", to_timestamp(col("timestamp")))

df_api.createOrReplaceTempView("weather_api")

total_event = df_api.count()
print(f"Total event cuaca terbaca dari HDFS: {total_event}")
df_api.printSchema()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/06 15:15:21 WARN Utils: Your hostname, Hadamard, resolves to a loopback address: 127.0.1.1; using 10.4.139.10 instead (on interface wlan0)
26/05/06 15:15:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 15:15:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Total event cuaca terbaca dari HDFS: 42
root
 |-- humidity: long (nullable = true)
 |-- kode_kota: string (nullable = true)
 |-- nama_kota: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- ts_obj: timestamp (nullable = true)



In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import hour, avg, max, min, col
from pyspark.sql.functions import round as spark_round # Untuk pembulatan hasil rata-rata suhu pakai spark_round()

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("WeatherAnalysis") \
    .getOrCreate()

# Gunakan port 8020 sesuai exposing Docker Hadoop Namenode Anda
df_api = spark.read.json(f"{HDFS_URI}/data/weather/api/")
print(df_api)

# suhu_per_kota = df_api.groupBy("nama_kota").agg(
#     avg("temperature").alias("suhu_avg"),
#     max("temperature").alias("suhu_tertinggi"),
#     min("temperature").alias("suhu_terendah")
# ).orderBy("suhu_avg", ascending=False)

# suhu_per_kota.show()
suhu_per_kota = (
    df_api.groupBy("kode_kota", "nama_kota")
    .agg(
        spark_round(avg("temperature"), 2).alias("suhu_avg"),
        max("temperature").alias("suhu_tertinggi"),
        min("temperature").alias("suhu_terendah"),
        count("*").alias("jumlah_event"),
    )
    .orderBy(col("suhu_avg").desc())
)

print("Analisis 1,Statistik suhu per kota")
suhu_per_kota.show(truncate=False)

top_suhu = suhu_per_kota.first()
if top_suhu:
    narasi_suhu = (
        f"{top_suhu['nama_kota']} memiliki rata-rata suhu tertinggi "
        f"({top_suhu['suhu_avg']} C). Kota ini perlu diprioritaskan untuk "
        "monitoring pengiriman pada jam panas."
    )
else:
    narasi_suhu = "Data suhu belum tersedia untuk dianalisis."
print("Interpretasi:", narasi_suhu)


26/05/06 15:15:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


DataFrame[humidity: bigint, kode_kota: string, nama_kota: string, temperature: double, timestamp: string, weather_code: bigint, wind_speed: double]
Analisis 1,Statistik suhu per kota
+---------+---------+--------+--------------+-------------+------------+
|kode_kota|nama_kota|suhu_avg|suhu_tertinggi|suhu_terendah|jumlah_event|
+---------+---------+--------+--------------+-------------+------------+
|SMG      |Semarang |32.0    |32.0          |32.0         |7           |
|SBY      |Surabaya |31.83   |31.9          |31.8         |7           |
|JKT      |Jakarta  |30.47   |30.6          |30.3         |7           |
|MKS      |Makassar |29.07   |29.1          |29.0         |7           |
|DPS      |Denpasar |28.66   |28.8          |28.6         |7           |
|MDN      |Medan    |26.87   |27.0          |26.7         |7           |
+---------+---------+--------+--------------+-------------+------------+

Interpretasi: Semarang memiliki rata-rata suhu tertinggi (32.0 C). Kota ini perlu dipr

In [14]:

# Cell 3 penjelasan terkait dengan 
from pyspark.sql.functions import count

# kondisi_ekstrem = df_api.filter(
#     (col("wind_speed")>40) | (col("humidity")>90) | (col("temperature")>35) 
# )

# rekap_ekstrem = kondisi_ekstrem.groupBy("nama_kota").agg(
#     count("*").alias("jumlah_event_ekstrem")
# ).orderBy("jumlah_event_ekstrem",ascending=False)

# rekap_ekstrem.show()
# print(f"Total event atau kejadian ekstrem : {kondisi_ekstrem.count()}")


kondisi_ekstrem = df_api.filter(
    (col("wind_speed") > 40) | (col("humidity") > 90) | (col("temperature") > 35)
)

rekap_ekstrem = (
    kondisi_ekstrem.groupBy("kode_kota", "nama_kota")
    .agg(count("*").alias("jumlah_event_ekstrem"))
    .orderBy(col("jumlah_event_ekstrem").desc())
)

total_ekstrem = kondisi_ekstrem.count()
print("\nAnalisis 2 - Deteksi kondisi cuaca ekstrem")
rekap_ekstrem.show(truncate=False)
print(f"Total event cuaca ekstrem: {total_ekstrem}")

top_ekstrem = rekap_ekstrem.first()
if top_ekstrem:
    narasi_ekstrem = (
        f"{top_ekstrem['nama_kota']} paling sering mengalami kondisi ekstrem "
        f"({top_ekstrem['jumlah_event_ekstrem']} event). Rute melalui kota ini "
        "perlu dicek ulang sebelum pengiriman."
    )
else:
    narasi_ekstrem = (
        "Tidak ada event ekstrem berdasarkan ambang wind_speed > 40 km/h, "
        "humidity > 90%, atau temperature > 35 C. Kondisi relatif aman pada "
        "data yang terkumpul."
    )
print("Interpretasi:", narasi_ekstrem)

# Resolving error pyspark 
#Hapus saja pemanggilan dengan wildcard seperti * 
# https://stackoverflow.com/questions/78583582/pyspark-sql-error-reading-csv-file-warn-filestreamsink-assume-no-metadata-dire


Analisis 2 - Deteksi kondisi cuaca ekstrem
+---------+---------+--------------------+
|kode_kota|nama_kota|jumlah_event_ekstrem|
+---------+---------+--------------------+
+---------+---------+--------------------+

Total event cuaca ekstrem: 0
Interpretasi: Tidak ada event ekstrem berdasarkan ambang wind_speed > 40 km/h, humidity > 90%, atau temperature > 35 C. Kondisi relatif aman pada data yang terkumpul.


In [ ]:
from pyspark.sql.functions import hour, avg

# Memperbaiki penulisan fungsi .agg() yang sebelumnya tertulis .groupBy("temperature").alias...
# tren_jam = df_api.withColumn("jam", hour(col("timestamp")))\
#     .groupBy("jam")\
#     .agg(avg("temperature").alias("suhu_avg"))\
#     .orderBy("jam")
    
# tren_jam.show(24)
HDFS_OUTPUT_PATH = f"{HDFS_URI}/data/weather/hasil"


# Pembenahan & koreksi ( 6 Mei 2026 By AtokTajuddin)
tren_jam = spark.sql(
    """
    SELECT
        HOUR(ts_obj) AS jam,
        ROUND(AVG(temperature), 2) AS suhu_avg,
        COUNT(*) AS jumlah_event
    FROM weather_api
    WHERE ts_obj IS NOT NULL
    GROUP BY HOUR(ts_obj)
    ORDER BY jam
    """
)

print("Analisis 3, Tren suhu rata-rata per jam")
tren_jam.show(24, truncate=False)

jam_terdingin = tren_jam.orderBy(col("suhu_avg").asc()).first()
if jam_terdingin:
    narasi_tren = (
        f"Jam {jam_terdingin['jam']:02d}:00 memiliki rata-rata suhu terendah "
        f"({jam_terdingin['suhu_avg']} C). Waktu ini dapat dipertimbangkan "
        "sebagai waktu pengiriman yang lebih nyaman."
    )
else:
    narasi_tren = "Data timestamp belum cukup untuk membaca tren suhu per jam."
print("Interpretasi:", narasi_tren)


# print("Menyimpan hasil analisis ke HDFS...")
# suhu_per_kota.write.mode("overwrite").json(f"{HDFS_OUTPUT_PATH}/suhu_per_kota")
# rekap_ekstrem.write.mode("overwrite").json(f"{HDFS_OUTPUT_PATH}/kondisi_ekstrem")
# tren_jam.write.mode("overwrite").json(f"{HDFS_OUTPUT_PATH}/tren_jam")
suhu_per_kota.write.mode("overwrite").json(f"{HDFS_OUTPUT_PATH}/suhu_per_kota")
rekap_ekstrem.write.mode("overwrite").json(f"{HDFS_OUTPUT_PATH}/kondisi_ekstrem")
tren_jam.write.mode("overwrite").json(f"{HDFS_OUTPUT_PATH}/tren_jam")

# Pengunaan write.mode sudah bisa dibilang best practice dan siap untuk production atau data skala besar

Analisis 3, Tren suhu rata-rata per jam
+---+--------+------------+
|jam|suhu_avg|jumlah_event|
+---+--------+------------+
|14 |29.83   |30          |
|15 |29.78   |12          |
+---+--------+------------+

Interpretasi: Jam 15:00 memiliki rata-rata suhu terendah (29.78 C). Waktu ini dapat dipertimbangkan sebagai waktu pengiriman yang lebih nyaman.


# **5. DATA SERVE**